In [1]:
# import required python modules
import os
import re
import unicodedata
import fitz
import pandas as pd

In [1]:
# set input and output paths
base_dir = r"D:\Egypt-Opportunity-Radar-Task3\data\raw\production"
in_path = os.path.join(base_dir, "task3_capmas_public_industrial_production_fy2023_24.pdf")
out_dir = os.path.join(base_dir, "cleaned")
os.makedirs(out_dir, exist_ok=True)

In [1]:
# open raw pdf document
doc = fitz.open(in_path)
records = []
page_summary = []

In [1]:
# extract activity table rows
for idx, page in enumerate(doc):
    text = unicodedata.normalize("NFKC", page.get_text("text"))
    lines = [ln.strip() for ln in text.splitlines() if ln.strip()]
    imgs = len(page.get_images())
    ptype = "picture only" if not lines and imgs > 0 else "single table page"
    page_summary.append({"page": idx + 1, "lines": len(lines), "images": imgs, "page_type": ptype})
    title = lines[0] if lines else ""
    for i, ln in enumerate(lines):
        m = re.match(r"^([A-Za-z][A-Za-z ,/()-]+?)\s+(\d{2,4})$", ln)
        if m:
            nums = []
            j = i + 1
            while j < len(lines) and re.match(r"^-?\d+(\.\d+)?$", lines[j]):
                nums.append(float(lines[j]))
                j += 1
            if nums:
                row = {"page": idx + 1, "table_heading": title, "isic_code": m.group(2), "activity_en": m.group(1).strip()}
                for k, val in enumerate(nums[:8], start=1):
                    row[f"col_{k}"] = val
                records.append(row)

In [1]:
# build cleaned dataframes
df_rec = pd.DataFrame(records)
df_pages = pd.DataFrame(page_summary)

In [1]:
# save cleaned public files
csv_out = os.path.join(out_dir, "cleaned_task3_capmas_public_industrial_production_fy2023_24.csv")
df_rec.to_csv(csv_out, index=False, encoding="utf-8-sig")

In [1]:
xlsx_out = os.path.join(out_dir, "cleaned_task3_capmas_public_industrial_production_fy2023_24.xlsx")
with pd.ExcelWriter(xlsx_out, engine="openpyxl") as writer:
    df_rec.to_excel(writer, sheet_name="activity_tables", index=False)
    df_pages.to_excel(writer, sheet_name="page_summary", index=False)